# Notebook exploring example training dataset

In [ ]:
from pathlib import Path

import numpy as np
import rasterio
import xarray as xr
from ipywidgets import widgets
from rasterio.control import GroundControlPoint
from rasterio.crs import CRS
from rasterio.io import MemoryFile
from rasterio.warp import Resampling, calculate_default_transform, reproject

xr.set_options(
    display_style="html",
    display_expand_data_vars=True,
    display_expand_attrs=True,
    display_expand_coords=True,
    display_expand_data=True,
)

## Read in example Ready-to-train and raw file

In [ ]:
data_toppath = Path(
    "/Users/gordonlogie/Documents/Projects/Internal_Projects/Prescient/arctic_showcase/data"
)
rtt_path = data_toppath.joinpath("autoice_dataset", "readytotrain", "21762830")
# rtt_path= data_toppath.joinpath("autoice_dataset", "readytotrain","demo")
raw_path = data_toppath.joinpath("autoice_dataset", "raw", "21762848")

data_outpath = data_toppath.joinpath("autoice_dataset", "data_exploration_outputs")
data_outpath.mkdir(parents=True, exist_ok=True)

In [ ]:
rtt_files = list(rtt_path.glob("*_prep.nc"))
rtt_label_files = list(rtt_path.glob("*_reference.nc"))
raw_files = list(raw_path.glob("*.nc"))

## Select image to explore

In [ ]:
im_select = widgets.Dropdown(
    options=[(f.stem, f) for f in sorted(rtt_files)], description="Select RTT file:"
)

display(im_select)

In [ ]:
# Open the selected RTT file and extract relevant information to
# find corresponding raw and label files
test_rtt_file = im_select.value
rtt_ds = xr.open_dataset(test_rtt_file)
original_id = rtt_ds.attrs["original_id"]
test_rtt_date = test_rtt_file.stem.split("_")[0]
test_raw_file = [f for f in raw_files if original_id in f.name][0]
test_label_file = [f for f in rtt_label_files if test_rtt_date in f.stem][0]

select_fname = test_rtt_file.stem
select_outpath = data_outpath.joinpath(f"{select_fname}")

In [ ]:
# Open the raw and label datasets
raw_ds = xr.open_dataset(test_raw_file)
rtt_label_ds = xr.open_dataset(test_label_file)

## Examine each dataset

In [ ]:
print(f"Raw Dataset: {test_raw_file.name}")
print("CRS Information:", raw_ds.crs if hasattr(raw_ds, "crs") else "no crs information available")
print("Number of variables:", len(raw_ds.data_vars))
var_list_raw = list(raw_ds.data_vars.keys())
raw_ds

In [ ]:
for var in var_list_raw:
    print(f"Variable: {var}, Dimensions: {raw_ds[var].dims}, Shape: {raw_ds[var].shape}")

In [ ]:
print(f"Ready-to-Train Dataset: {test_rtt_file.name}")
print("CRS Information:", rtt_ds.crs if hasattr(rtt_ds, "crs") else "no crs information available")
print("Number of variables:", len(rtt_ds.data_vars))
var_list_rtt = list(rtt_ds.data_vars.keys())
rtt_ds

In [ ]:
for var in var_list_rtt:
    print(f"Variable: {var}, Dimensions: {rtt_ds[var].dims}, Shape: {rtt_ds[var].shape}")

In [ ]:
print("Ready-to-Train Label Dataset:", test_label_file.name)
print("CRS Information:", rtt_ds.crs if hasattr(rtt_ds, "crs") else "no crs information available")
print("Number of variables:", len(rtt_ds.data_vars))
var_list_rtt_label = list(rtt_label_ds.data_vars.keys())
rtt_label_ds

In [ ]:
for var in var_list_rtt_label:
    print(f"""Variable: {var}, Dimensions: {rtt_label_ds[var].dims}, 
          Shape: {rtt_label_ds[var].shape}""")

In [ ]:
missing_in_rtt = set(var_list_raw) - set(var_list_rtt)
print("Variables in raw dataset but missing in ready-to-train dataset:")
for var in missing_in_rtt:
    print(f"- {var}")

missing_in_raw = set(var_list_rtt) - set(var_list_raw)
print("\nVariables in ready-to-train dataset but missing in raw dataset:")
for var in missing_in_raw:
    print(f"- {var}")

## Process Raw Sea Ice Charts
For the raw dataset, create new spatial variables by combining the polygon_icechart variable and polygon_codes look up table

In [ ]:
# Lookup tables from AI4Arctic utils.py (https://github.com/astokholm/AI4ArcticSeaIceChallenge)
SIC_LOOKUP = {
    "polygon_idx": 0,
    "total_sic_idx": 1,
    "sic_partial_idx": [2, 5, 8],
    0: 0,
    1: 0,
    2: 0,
    55: 0,
    10: 1,
    20: 2,
    30: 3,
    40: 4,
    50: 5,
    60: 6,
    70: 7,
    80: 8,
    90: 9,
    91: 10,
    92: 10,
    "mask": 255,
    "n_classes": 12,
}
SOD_LOOKUP = {
    "sod_partial_idx": [3, 6, 9],
    "threshold": 0.65,
    "invalid": -9,
    "water": 0,
    0: 0,
    80: 0,
    81: 1,
    82: 1,
    83: 2,
    84: 2,
    85: 2,
    86: 4,
    87: 3,
    88: 3,
    89: 3,
    91: 4,
    93: 4,
    95: 5,
    96: 5,
    97: 5,
    98: 255,
    99: 255,
    "mask": 255,
    "n_classes": 7,
}
FLOE_LOOKUP = {
    "floe_partial_idx": [4, 7, 10],
    "threshold": 0.65,
    "invalid": -9,
    "water": 0,
    0: 0,
    22: 255,
    1: 255,
    2: 1,
    3: 2,
    4: 3,
    5: 4,
    6: 5,
    7: 5,
    8: 255,
    9: 6,
    10: 6,
    21: 255,
    "fastice_class": 255,
    "mask": 255,
    "n_classes": 8,
}
ICECHART_NOT_FILLED_VALUE = -9
ICECHART_UNKNOWN = 99

# Parse polygon_codes: entry 0 is the header row ('poly_id;CT;CA;SA;FA;...');
# entries 1..n are the actual polygon records.
# Stack gives shape (n_entries, n_fields=15); skip header with [1:].
codes_raw = np.stack(np.char.split(raw_ds["polygon_codes"].values.astype(str), sep=";"), 0)
poly_type_col = codes_raw[1:, -1]  # POLY_TYPE string per polygon ('I' or 'O'/'W')
codes = codes_raw[1:, :-2].astype(int)  # drop CF and POLY_TYPE cols; shape (n_polygons, 13)
# Column layout after drop: poly_id(0), CT(1), CA(2), SA(3), FA(4), CB(5), SB(6), FB(7),
#                           CC(8), SC(9), FC(10), CN(11), CD(12)

sic_partial = SIC_LOOKUP["sic_partial_idx"]  # [2, 5, 8] → CA, CB, CC
sod_idx = SOD_LOOKUP["sod_partial_idx"]  # [3, 6, 9] → SA, SB, SC
floe_idx = FLOE_LOOKUP["floe_partial_idx"]  # [4, 7, 10] → FA, FB, FC
tot_idx = SIC_LOOKUP["total_sic_idx"]

# Convert SIGRID-3 codes to class values for SIC (CT and partial CA/CB/CC).
converted_codes = codes.copy().astype(float)
for key, value in SIC_LOOKUP.items():
    if isinstance(key, int):
        for col in sic_partial:
            converted_codes[converted_codes[:, col] == key, col] = value
        converted_codes[converted_codes[:, tot_idx] == key, tot_idx] = value

# When CT has ice but CA is empty, copy CT class into CA
# so dominance check has something to work with.
ice_ct_ca_empty = (converted_codes[:, tot_idx] > SIC_LOOKUP[0]) & (
    converted_codes[:, sic_partial[0]] == ICECHART_NOT_FILLED_VALUE
)
converted_codes[ice_ct_ca_empty, sic_partial[0]] = converted_codes[ice_ct_ca_empty, tot_idx]

# Convert SIGRID-3 codes to class values for SOD partial columns (SA, SB, SC).
for key, value in SOD_LOOKUP.items():
    if isinstance(key, int):
        for col in sod_idx:
            converted_codes[converted_codes[:, col] == key, col] = value

# Convert SIGRID-3 codes to class values for FLOE partial columns (FA, FB, FC).
for key, value in FLOE_LOOKUP.items():
    if isinstance(key, int):
        for col in floe_idx:
            converted_codes[converted_codes[:, col] == key, col] = value

converted_codes = converted_codes.astype(int)

# When two or more partial layers share the same SOD/FLOE class, sum their partial SIC
# concentrations so the dominance threshold check is applied to the combined class coverage.
sod_pairs = [[0, 1], [0, 2], [1, 2]]
sod_partial_add = np.zeros_like(converted_codes)
floe_partial_add = np.zeros_like(converted_codes)

for i, j in sod_pairs:
    sod_match = (converted_codes[:, sod_idx[i]] == converted_codes[:, sod_idx[j]]) & (
        converted_codes[:, sod_idx[i]] != ICECHART_NOT_FILLED_VALUE
    )
    floe_match = (converted_codes[:, floe_idx[i]] == converted_codes[:, floe_idx[j]]) & (
        converted_codes[:, floe_idx[i]] != ICECHART_NOT_FILLED_VALUE
    )
    sod_partial_add[sod_match, sic_partial[i]] += converted_codes[sod_match, sic_partial[j]]
    floe_partial_add[floe_match, sic_partial[i]] += converted_codes[floe_match, sic_partial[j]]

tmp_sod = converted_codes + sod_partial_add
tmp_floe = converted_codes + floe_partial_add

# Capture the NaN mask from polygon_icechart before conversion.
# Over-land and outside-chart pixels are float NaN in the raw file; np.nan cast to uint8
# silently becomes 0 (open water), so we preserve them explicitly as fill value 255.
scene_tmp = raw_ds["polygon_icechart"].values.copy()
nan_mask = np.isnan(scene_tmp)
print(f"NaN pixels in polygon_icechart: {nan_mask.sum():,} ({nan_mask.mean() * 100:.1f}%)")

# Build per-pixel output arrays by iterating over polygons and looking up their pixel locations.
sic = scene_tmp.copy()
sod = scene_tmp.copy()
floe = scene_tmp.copy()

with np.errstate(divide="ignore", invalid="ignore"):
    for i in range(converted_codes.shape[0]):
        px = np.where(scene_tmp == converted_codes[i, SIC_LOOKUP["polygon_idx"]])
        sic[px] = converted_codes[i, tot_idx]

        if np.char.lower(poly_type_col[i]) == "w":
            sic[px] = SIC_LOOKUP[0]

        ct = tmp_sod[i, tot_idx]
        if np.divide(np.max(tmp_sod[i, sic_partial]), ct) * 100 >= SOD_LOOKUP["threshold"] * 100:
            sod[px] = converted_codes[i, sod_idx[np.argmax(tmp_sod[i, sic_partial])]]
        else:
            sod[px] = ICECHART_NOT_FILLED_VALUE

        if np.divide(np.max(tmp_floe[i, sic_partial]), ct) * 100 >= FLOE_LOOKUP["threshold"] * 100:
            floe[px] = converted_codes[i, floe_idx[np.argmax(tmp_floe[i, sic_partial])]]
        else:
            floe[px] = ICECHART_NOT_FILLED_VALUE

        if any(converted_codes[i, floe_idx] == FLOE_LOOKUP["fastice_class"]):
            floe[px] = FLOE_LOOKUP["fastice_class"]

# Mask ambiguous polygons and unknown codes.
sod[sod == SOD_LOOKUP["invalid"]] = SOD_LOOKUP["mask"]
floe[floe == FLOE_LOOKUP["invalid"]] = FLOE_LOOKUP["mask"]
for arr in (sic, sod, floe):
    arr[arr == ICECHART_UNKNOWN] = 255

# Enforce open-water consistency across all three charts.
sod[sic == SIC_LOOKUP[0]] = SOD_LOOKUP["water"]
floe[sic == SIC_LOOKUP[0]] = FLOE_LOOKUP["water"]

# Propagate the original NaN mask as fill value 255 so over-land and outside-chart
# pixels are masked rather than silently becoming class 0 on uint8 cast.
sic[nan_mask] = 255
sod[nan_mask] = 255
floe[nan_mask] = 255

dims = raw_ds["polygon_icechart"].dims
raw_ds = raw_ds.assign(
    {
        "SIC_derived": xr.DataArray(
            sic.astype(np.uint8),
            dims=dims,
            attrs={
                "long_name": "Sea Ice Concentration derived from polygon_codes CT field",
                "chart_fill_value": 255,
            },
        ),
        "SOD_derived": xr.DataArray(
            sod.astype(np.uint8),
            dims=dims,
            attrs={
                "long_name": "Stage of Development derived from polygon_codes SA field",
                "chart_fill_value": 255,
            },
        ),
        "FLOE_derived": xr.DataArray(
            floe.astype(np.uint8),
            dims=dims,
            attrs={
                "long_name": "Floe Size derived from polygon_codes FA field",
                "chart_fill_value": 255,
            },
        ),
    }
)

for name in ("SIC_derived", "SOD_derived", "FLOE_derived"):
    da = raw_ds[name]
    unique = np.unique(da.values)
    print(f"{name}: shape={da.shape}, unique values={unique}")

In [ ]:
raw_ds

## Georeference and output data variables
This section allows a user to output a select variable from either the raw or RTT dataset as a georeferenced geotiff. Images can be output in Lat/lon (WGS84), EPSG:3978 — NAD83 / Canada Atlas Lambert, or a scene-native UTM projection

In [ ]:
SAR_DIMS = {"sar_lines", "sar_samples"}
TWO_KM_DIMS = {"2km_grid_lines", "2km_grid_samples"}

_WGS84_PROJ = "+proj=longlat +datum=WGS84 +no_defs"
# NAD83 / Canada Atlas Lambert
_EPSG3978_PROJ = (
    "+proj=lcc +lat_0=63.390675 +lon_0=-91.8666666666667 +lat_1=49 +lat_2=77"
    " +x_0=6200000 +y_0=3000000 +ellps=GRS80 +towgs84=0,0,0,0,0,0,0 +units=m "
    "+no_defs"
)


def _scene_native_proj(gcps):
    """Derive a UTM PROJ string from the centroid of the GCP cloud."""
    center_lon = sum(g.x for g in gcps) / len(gcps)
    center_lat = sum(g.y for g in gcps) / len(gcps)
    zone = int((center_lon + 180) / 6) + 1
    hemisphere = "north" if center_lat >= 0 else "south"
    return f"+proj=utm +zone={zone} +{hemisphere} +datum=WGS84 +units=m +no_defs"


def _nodata_for_dtype(dtype):
    """NaN for floats (matches existing masked pixels);
    dtype max for integers (e.g. 255 for uint8)."""
    if np.issubdtype(dtype, np.floating):
        return np.nan
    return int(np.iinfo(dtype).max)


def project_and_output_variable_as_geotiff(
    select_fname, dataset_name, dataset, var, out_path, projection="wgs84"
):
    da = dataset[var]

    if set(da.dims) not in (SAR_DIMS, TWO_KM_DIMS):
        print(f"Skipping {var}: no georeferencing available for dims {da.dims}")
        return

    data = da.values
    is_2km = set(da.dims) == TWO_KM_DIMS
    nodata = _nodata_for_dtype(data.dtype)

    if "sar_grid_line" in dataset.data_vars:
        lines = dataset["sar_grid_line"].values
        samples = dataset["sar_grid_sample"].values
        lats = dataset["sar_grid_latitude"].values
        lons = dataset["sar_grid_longitude"].values
        if is_2km:
            scale_l = data.shape[0] / dataset.sizes["sar_lines"]
            scale_s = data.shape[1] / dataset.sizes["sar_samples"]
            gcps = [
                GroundControlPoint(
                    row=lines[i] * scale_l, col=samples[i] * scale_s, x=lons[i], y=lats[i]
                )
                for i in range(len(lines))
            ]
        else:
            gcps = [
                GroundControlPoint(row=lines[i], col=samples[i], x=lons[i], y=lats[i])
                for i in range(len(lines))
            ]
    else:
        lats = dataset["sar_grid2d_latitude"].values
        lons = dataset["sar_grid2d_longitude"].values
        n_dim0, n_dim1 = lats.shape
        # First array dim is lines (rows), second is samples (cols) — naming is misleading
        line_pos = np.linspace(0, data.shape[0] - 1, n_dim0)
        sample_pos = np.linspace(0, data.shape[1] - 1, n_dim1)
        gcps = [
            GroundControlPoint(row=line_pos[i], col=sample_pos[j], x=lons[i, j], y=lats[i, j])
            for i in range(n_dim0)
            for j in range(n_dim1)
        ]

    # Bypass EPSG lookup (broken proj.db from conda) — use PROJ string directly
    wgs84 = CRS.from_string(_WGS84_PROJ)

    out_dir = out_path.joinpath(dataset_name, projection)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / f"{select_fname}_{dataset_name}_{var}_{projection}.tif"

    if projection == "wgs84":
        with rasterio.open(
            out_file,
            "w",
            driver="GTiff",
            height=data.shape[0],
            width=data.shape[1],
            count=1,
            dtype=data.dtype,
            crs=None,
            nodata=nodata,
        ) as dst:
            dst.write(data, 1)
            dst.gcps = (gcps, wgs84)
        print(f"Written to {out_file}")
        print(f"GCPs embedded: {len(gcps)}")
    else:
        if projection == "epsg3978":
            dst_crs = CRS.from_string(_EPSG3978_PROJ)
        else:  # scene_native
            dst_crs = CRS.from_string(_scene_native_proj(gcps))

        # calculate_default_transform with gcps= triggers a rasterio VRT bug (both <SRS> and
        # <GCPList> set), so derive the source bounds from GCP extents instead.
        gcp_lons = [g.x for g in gcps]
        gcp_lats = [g.y for g in gcps]
        transform, dst_width, dst_height = calculate_default_transform(
            wgs84,
            dst_crs,
            data.shape[1],
            data.shape[0],
            left=min(gcp_lons),
            bottom=min(gcp_lats),
            right=max(gcp_lons),
            top=max(gcp_lats),
        )

        with MemoryFile() as memfile:
            with memfile.open(
                driver="GTiff",
                height=data.shape[0],
                width=data.shape[1],
                count=1,
                dtype=data.dtype,
            ) as src:
                src.write(data, 1)
                src.gcps = (gcps, wgs84)

                with rasterio.open(
                    out_file,
                    "w",
                    driver="GTiff",
                    height=dst_height,
                    width=dst_width,
                    count=1,
                    dtype=data.dtype,
                    crs=dst_crs,
                    transform=transform,
                    nodata=nodata,
                ) as dst:
                    reproject(
                        source=rasterio.band(src, 1),
                        destination=rasterio.band(dst, 1),
                        gcps=gcps,
                        src_crs=wgs84,
                        dst_crs=dst_crs,
                        dst_transform=transform,
                        resampling=Resampling.nearest,
                        dst_nodata=nodata,
                    )

        print(f"Written to {out_file}")
        print(f"Reprojected to: {dst_crs.to_string()}")

### Add lat and lon from RTT to RTT Labels

In [ ]:
rtt_label_ds = rtt_label_ds.assign(
    {
        "sar_grid2d_latitude": rtt_ds["sar_grid2d_latitude"],
        "sar_grid2d_longitude": rtt_ds["sar_grid2d_longitude"],
    }
)

### Select Dataset to output
RTT labels are appended to RTT dataset to output

In [ ]:
datasets = {
    "Raw Dataset": raw_ds,
    "Ready-to-Train Dataset": rtt_ds,
    "Ready-to-Train Label Dataset": rtt_label_ds,
}

ds_select = widgets.Dropdown(
    options=list(datasets.keys()),
    description="Select Dataset:",
)

variable_select = widgets.SelectMultiple(
    options=list(datasets[ds_select.value].data_vars),
    description="Select Variable(s):",
    value=list(datasets[ds_select.value].data_vars),
)

projection_select = widgets.Dropdown(
    options=[
        ("WGS84 (unprojected, GCPs embedded)", "wgs84"),
        ("Arctic / Canada Atlas Lambert (EPSG:3978)", "epsg3978"),
        ("Scene native (UTM)", "scene_native"),
    ],
    value="wgs84",
    description="Projection:",
)


def update_variables(change):
    variable_select.options = list(datasets[change["new"]].data_vars)
    variable_select.value = list(datasets[change["new"]].data_vars)


ds_select.observe(update_variables, names="value")

widgets.VBox([ds_select, variable_select, projection_select])

In [ ]:
select_vars = variable_select.value
select_ds_name = ds_select.value
select_projection = projection_select.value

for select_var in select_vars:
    project_and_output_variable_as_geotiff(
        select_fname=select_fname,
        dataset_name=select_ds_name.replace(" ", "_"),
        dataset=datasets[select_ds_name],
        var=select_var,
        out_path=select_outpath,
        projection=select_projection,
    )